# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Logistic Regression first, then Random Forest. `is_declining_label` is an observed proxy label (not clustering, not "what drives X"), and the deliverable is a ranking ("which page first?") -- the toolkit says start readable (Logistic Regression), add a stronger model (Random Forest) only if it earns its complexity, and evaluate any classifier's probability at precision@K rather than accuracy. Both models score every page with `predict_proba`; the same `precision_at_k` used on the Week-4 baseline's raw rule score applies unchanged to a model's probability -- same evaluation, different scorer.

In [1]:
import sys
sys.path.insert(0, "scripts")
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("rows:", len(df), "| declining rate:", round(df['is_declining_label'].mean(), 3))


rows: 30000 | declining rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Same client-holdout split as the ML-07 baseline: `GroupShuffleSplit(test_size=0.25, random_state=42)` grouped on `client_id` -- same seed, so Section 3 scores the model on the identical 7,115 test pages the baseline was evaluated on, not a fresh split that could flatter either side. Grouped, not row-level random, so no client's pages appear in both halves -- a model that just memorized one client's typical page wouldn't be caught by a row-level split, only a client-level one.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
print("train pages:", len(train_idx), "| train clients:", df.iloc[train_idx]["client_id"].nunique())
print("test pages:", len(test_idx), "| test clients:", df.iloc[test_idx]["client_id"].nunique())


train pages: 22885 | train clients: 24
test pages: 7115 | test clients: 8


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Features are exactly the ML-04 **Feature** bucket: `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` from `scripts/ml_utils.py`, plus `log1p` of the raw traffic totals (heavy-tailed) and `has_*` missingness flags instead of a blind `fillna(0)` -- ML-04 showed `search_volume`/`word_count` go blank by `content_type`, so a silent zero would encode content type into the features. The baseline's own rule score (frozen from ML-07, not refit) sits in the same comparison table, scored on the same test pages.

In [3]:
# has_* flags before filling -- missingness tracks content_type (ML-04), a flag keeps that honest
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = list(MODEL_NUMERIC_FEATURES) + ["has_keyword_data", "has_word_count", "has_scroll_data"]
for col in numeric_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_features = list(MODEL_CATEGORICAL_FEATURES)
for col in categorical_features:
    df[col] = df[col].fillna("unknown").astype(str)

X = pd.concat([df[numeric_features], pd.get_dummies(df[categorical_features], prefix=categorical_features)], axis=1)
y = df["is_declining_label"]

# ML-07 baseline, recomputed identically (frozen rule, not refit)
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
logreg.fit(X_train, y_train)
p_log = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

baseline_test_scores = df.iloc[test_idx]["baseline_score"]
base_rate = float(y_test.mean())

comparison = pd.DataFrame({
    "model": ["baseline_rules", "logistic_regression", "random_forest"],
    "roc_auc": [None, roc_auc_score(y_test, p_log), roc_auc_score(y_test, p_rf)],
    "avg_precision": [None, average_precision_score(y_test, p_log), average_precision_score(y_test, p_rf)],
    "precision_at_20": [
        precision_at_k(y_test, baseline_test_scores, 20),
        precision_at_k(y_test, p_log, 20),
        precision_at_k(y_test, p_rf, 20),
    ],
    "precision_at_50": [
        precision_at_k(y_test, baseline_test_scores, 50),
        precision_at_k(y_test, p_log, 50),
        precision_at_k(y_test, p_rf, 50),
    ],
})
comparison = comparison.round(3)
print(f"base rate (test, {len(y_test)} pages, {df.iloc[test_idx]['client_id'].nunique()} clients): {base_rate:.3f}\n")
print(comparison.to_string(index=False))

Path("work/outputs").mkdir(parents=True, exist_ok=True)
records = comparison.to_dict(orient="records")
for record in records:
    for key, value in record.items():
        if isinstance(value, float) and np.isnan(value):
            record[key] = None

with open("work/outputs/model_comparison.json", "w") as f:
    json.dump({"base_rate_test": round(base_rate, 3), "comparison": records}, f, indent=2)


base rate (test, 7115 pages, 8 clients): 0.517

              model  roc_auc  avg_precision  precision_at_20  precision_at_50
     baseline_rules      NaN            NaN            0.589            0.545
logistic_regression    0.610          0.605            0.800            0.700
      random_forest    0.603          0.587            0.550            0.560


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Logistic Regression wins both the tie-aware baseline and Random Forest at Precision@20 (0.80 vs baseline 0.589, RF 0.55) and Precision@50 (0.70 vs baseline 0.545, RF 0.56) -- report that as the finding rather than defaulting to the fancier model. The baseline metric uses the expected hit rate when its zero-score tie crosses K, so CSV order and pandas versions cannot change the comparison. ROC AUC (0.610 vs 0.603) and average precision (0.605 vs 0.587) are close between the two models; the gap that matters for this decision shows up specifically at the top of the ranked queue, which is exactly what the editors act on.

**Where it's wrong:**
- **False positives (15 of the predicted top 50) skew toward pages that are actually trending `up` or `stable`, not `down`.** Several sit at moderate traffic (100-300 impressions/90d) with mid-pack position (6-31) and are 100-275 days old -- they *look* like a typical decliner on every pre-decision signal available, but the direction was wrong. Without trend history (excluded as leakage, ML-04), the model can't distinguish "aging and about to slip" from "aging and holding steady" -- both produce the same feature pattern today.
- **False negatives cluster hard on very-low-traffic pages** (1-10 impressions/90d, position 2-45, 310-545 days old) that genuinely are declining but carry almost no signal to work with: with `days_with_impressions` and `log_impressions_90d` as the top two features, a page this quiet reads as "nothing happening" everywhere, not specifically "declining." The model's confidence correctly tracks how much evidence exists, not just the label -- but that means it systematically misses the quietest decliners.
- **Hit rate is slightly lower on `comparison article` (0.625) than `keyword article` (0.735)** within the predicted top 50 -- a real but modest gap, not a red flag on its own; worth re-checking for a sample-size effect (16 vs 34 pages) before reading it as a content-type bias.

**What it leans on:** Random Forest's top features -- `days_with_impressions` (0.149), `log_impressions_90d` (0.130), `avg_position` (0.103), `content_age_days` (0.087) -- and Logistic Regression's largest coefficients -- `log_impressions_90d` (+1.49), `impression_tier_low` (+0.83), `position_tier_top_3` (-0.79) -- all make sense: consistent visibility, position, and age are plausible correlates of a page's trajectory. None is suspiciously dominant (RF's top feature is 0.149 of 1.0, not 0.9+), which is the sanity check that matters -- a single feature owning almost all the importance is usually leakage, and that's not what this shows.

In [4]:
test_df = df.iloc[test_idx].copy()
test_df["model_prob"] = p_log

predicted_top50 = test_df.sort_values("model_prob", ascending=False).head(50)
false_positives = predicted_top50[predicted_top50["is_declining_label"] == 0]
print(f"false positives in predicted top-50: {len(false_positives)} / 50")
print(false_positives[["model_prob", "impressions_90d", "avg_position", "content_age_days", "trend_direction", "content_type"]].head(5).to_string(index=False))

lowest_ranked_decliners = test_df[test_df["is_declining_label"] == 1].sort_values("model_prob").head(5)
print("\nlowest-ranked pages that ARE declining (false negatives):")
print(lowest_ranked_decliners[["model_prob", "impressions_90d", "avg_position", "content_age_days", "trend_direction", "content_type"]].to_string(index=False))

print("\nhit rate within predicted top-50, by content_type:")
print(predicted_top50.groupby("content_type")["is_declining_label"].agg(["count", "mean"]).round(3))

coefs = pd.Series(logreg.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("\ntop 8 logistic regression coefficients by |weight|:")
print(coefs.head(8).round(3))

rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\ntop 8 random forest feature importances:")
print(rf_importance.head(8).round(3))


false positives in predicted top-50: 15 / 50
 model_prob  impressions_90d  avg_position  content_age_days trend_direction       content_type
   0.960450              290           5.9                96              up    keyword article
   0.937456              181           7.2               140              up comparison article
   0.935928              235          31.0               181          stable    keyword article
   0.935392             3115          12.8               275          stable    keyword article
   0.924401              102           7.2               225              up comparison article

lowest-ranked pages that ARE declining (false negatives):
 model_prob  impressions_90d  avg_position  content_age_days trend_direction    content_type
   0.045414                2          45.0               537            down keyword article
   0.054181                3           2.0               545            down keyword article
   0.063728                3          41.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.